# Variable-Channel Spatial ASR — 학습 템플릿 노트북

화자 구분 비교 사다리의 모든 시스템을 이 노트북 하나로 학습·검증합니다. 모델마다 노트북을 복사하지 않고
맨 위 `CONFIG`만 바꿉니다. 모델 구조는 저장소의 `models/`, 학습 방법은 `training/`, 모델별 설정은
`configs/<이름>.json`에 있고, 이 노트북은 그 코드를 불러 실행하기만 합니다.

| CONFIG | 시스템 | 학습 여부 |
|---|---|---|
| `smoke` | 코드 경로 점검용 초소형 제안 모델 (가짜 음성 · 간이 RIR · 대체 인코더, 몇 분) | 학습 (의미 없는 숫자) |
| `smoke_real` | GPU 점검: 실제 ECAPA · pyroomacoustics · 실제 크기 모델로 30걸음 (dev-clean만, 약 10분) | 학습 (의미 없는 숫자) |
| `voice_only` | 목소리 전용: 동결 ECAPA + 온라인 군집 | 임계값만 조정 |
| `space_only` | 공간 전용: SRP-PHAT 방위 분포 + 온라인 군집 | 임계값만 조정 |
| `naive_fusion` | 단순 결합: 두 유사도를 고정 가중치로 합산 | 가중치·임계값만 조정 |
| `proposed_carry` | 제안 모델: TAC + ECAPA + GRU, 청크 간 상태 유지 (Carry-Over) | 학습 |
| `proposed_reset` | 제안 모델: 같은 구조, 청크마다 상태 초기화 (Chunk-Reset) | 학습 |

체크포인트·지표·요약은 Google Drive에 저장됩니다. 런타임이 끊기면 노트북을 위에서부터 다시 실행하면
마지막 체크포인트에서 이어서 학습합니다. 마지막 셀이 출력하는 요약 블록을 그대로 붙여 넣어 주시면 됩니다.

저장소: https://github.com/hd1534/variable-channel-spatial-asr

## 0. 설정

`CONFIG`로 시스템을 고릅니다. 처음에는 `smoke_real`을 GPU 런타임에서 한 번 돌려 주세요. 설치, 체크포인트 저장·재개,
혼합 정밀도, GPU 메모리, 데이터 준비 시간(`data_sec`) 대 GPU 계산 시간(`step_sec`)을 실제 조건에서 확인합니다.

In [ ]:
CONFIG = "smoke_real"  # @param ["smoke", "smoke_real", "voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop"]
# 학습 화자. train-clean-100(251명, 6.3 GB)이 기본입니다. dev-clean-only는 큰 내려받기 없이
# dev-clean의 평가 제외 화자 34명을 학습 24명 · 검증 10명으로 나눠 빨리 돌려 보는 용도입니다.
SPEAKERS = "train-clean-100"  # @param ["train-clean-100", "dev-clean-only"]
# 설정 덮어쓰기 (공백으로 구분). 예: train.max_steps=2000 train.batch_size=8
# 이미 돌린 CONFIG의 설정을 바꾸면 이어서 학습할 수 없으므로 RUN_TAG로 새 실행 폴더를 만드세요.
OVERRIDES = ""  # @param {type:"string"}
RUN_TAG = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
# 특정 커밋으로 고정하려면 커밋 해시를 적습니다 (비우면 BRANCH의 최신).
COMMIT = ""  # @param {type:"string"}
# 체크포인트를 만든 커밋과 지금 코드가 달라도 이어서 학습하려면 True (쓴 커밋은 모두 기록됩니다).
ALLOW_CODE_CHANGE = False  # @param {type:"boolean"}

REPO_URL = "https://github.com/hd1534/variable-channel-spatial-asr.git"
DRIVE_ROOT = "/content/drive/MyDrive/vcsasr"  # 체크포인트 · 데이터 캐시 (런타임이 끊겨도 남는다)
DATA_ROOT = "/content/data"                   # 학습 중 읽는 데이터 (로컬 디스크가 Drive보다 훨씬 빠르다)
REPO_DIR = "/content/variable-channel-spatial-asr"

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path


def sh(cmd, check=True):
    """셸 명령을 실행하고 출력을 셀에 바로 보여 준다. 실패하면 셀을 멈춘다."""
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")  # 자식 파이썬의 로그가 쌓이지 않고 바로 보이게
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0 and check:
        raise RuntimeError(f"실패 (종료 코드 {proc.returncode}): {cmd}")
    return proc.returncode


IN_COLAB = "google.colab" in sys.modules
import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"Colab: {IN_COLAB} | torch {torch.__version__} | GPU: {GPU or '없음'}"
      + (f" | compute capability {torch.cuda.get_device_capability()}" if GPU else ""))
if IN_COLAB and GPU is None and CONFIG.startswith("proposed"):
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고른 뒤 다시 실행하세요.")

## 1. Google Drive 연결과 저장소 받기

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

if not Path(REPO_DIR, ".git").is_dir():
    # 평가셋 오디오(data/*/audio, 약 1.3 GB)는 학습에 쓰지 않으므로 받지 않는다.
    # 화자 누수 검사가 읽는 data/ 의 텍스트 파일은 그대로 받는다.
    sh(f'git clone -q --depth 1 --filter=blob:none --no-checkout -b {BRANCH} {REPO_URL} "{REPO_DIR}"')
    sh(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/*/audio/'")
sh(f'git -C "{REPO_DIR}" fetch -q --depth 1 origin {COMMIT or BRANCH} && git -C "{REPO_DIR}" checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
sh("git log -1 --format='%h %s'")

## 2. 패키지 설치

Colab에 미리 설치된 `torch`·`torchaudio`·`numpy`는 그대로 두고(제약 파일로 고정), 동결 화자 인코더(ECAPA-TDNN)를
불러올 `speechbrain`만 설치합니다. 방 음향을 계산하는 `pyroomacoustics`는 Drive에 방 음향 뱅크가 없을 때만 3절에서
설치합니다.

In [ ]:
sh("pip freeze | grep -iE '^(torch|torchaudio|numpy)==' > /content/constraints.txt")
sh('pip -q install -c /content/constraints.txt "speechbrain==1.1.2"')
sh(f'{sys.executable} -c "import torchaudio, soundfile; from speechbrain.inference.speaker import EncoderClassifier; '
   f'import torch; print(\'torch\', torch.__version__, \'| speechbrain 설치 확인\')"')

## 3. 데이터 준비

- **LibriSpeech**: 압축 파일을 Drive(`vcsasr/cache/`)에 한 번 받아 두고, 매 세션 로컬 디스크로 풉니다.
  OpenSLR이 공개한 md5와 대조합니다. 학습 화자는 train-clean-100, 검증 화자는 dev-clean 중
  **사전 측정 평가셋 화자 6명을 뺀** 34명입니다. 본 평가셋 화자는 test-clean 40명이고, 학습·검증 코퍼스에
  이들이 섞여 있으면 학습이 시작되지 않습니다. 그래서 평가에 쓰는 목소리는 학습에도 임계값 조정에도 쓰이지 않습니다.
- **방 음향 뱅크**: pyroomacoustics로 방 800개를 한 번 계산해 Drive(`vcsasr/rirbanks/`, 약 1.2 GB)에 둡니다. 방마다 마이크 4개,
  음원 위치 4곳이고, 위치마다 0.1~0.3 m 옮긴 사본이 2개 더 있습니다. 학습 세션은 이 방들에 매번 새 음성을 섞어 만듭니다.
  평가셋과 같은 버전(0.8.5)이어야 하는데 Linux용 설치 파일이 없어서, 뱅크가 없을 때만 Git 태그에서 빌드합니다(약 3분).
  방 하나에 코어 하나로 평균 3초쯤(방마다 0.1~16초) 걸려, 코어가 12개면 몇 분, 2개면 한 시간 안팎 걸립니다. 50개 방이 끝날 때마다
  Drive에 바로 복사되므로 끊겨도 이어서 만듭니다. 학습하지 않는 시스템은 검증용 방 64개만 씁니다.
- **ECAPA 체크포인트**: Drive(`vcsasr/models/ecapa/`)에 저장해 다음 세션부터는 받지 않습니다.

In [ ]:
LIBRISPEECH_MD5 = {
    "dev-clean": "42e2234ba48799c1f50f24a7926300a1",
    "train-clean-100": "2a93770f6d5c6c964bc36631d331a522",
}
MIRRORS = ["https://www.openslr.org/resources/12", "https://openslr.elda.org/resources/12",
           "https://openslr.trmal.net/resources/12"]
cfg_json = json.loads(Path(f"configs/{CONFIG}.json").read_text(encoding="utf-8"))
TRAINABLE = cfg_json.get("model", {}).get("name", "proposed") == "proposed"
SYNTHETIC = bool(cfg_json.get("data", {}).get("synthetic"))  # smoke: 가짜 음성이라 코퍼스가 필요 없다
OWN_TRAIN_ROOT = "train_root" in cfg_json.get("data", {}) or SYNTHETIC  # 설정이 학습 화자를 직접 정한다
need = [] if SYNTHETIC else ["dev-clean"]
if TRAINABLE and SPEAKERS == "train-clean-100" and not OWN_TRAIN_ROOT:
    need.append("train-clean-100")

cache = Path(DRIVE_ROOT, "cache")
cache.mkdir(parents=True, exist_ok=True)
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
for name in need:
    if Path(DATA_ROOT, "LibriSpeech", name).is_dir():
        print("이미 풀려 있음:", name)
        continue
    cached = cache / f"{name}.tar.gz"  # md5를 통과한 파일만 이 이름으로 저장된다
    if cached.is_file():
        tar = cached
    else:
        tar = Path(DATA_ROOT, f"{name}.tar.gz")
        for mirror in MIRRORS:
            if sh(f'curl -L --fail --retry 3 -C - -o "{tar}" {mirror}/{name}.tar.gz', check=False) == 0:
                break
        else:
            raise RuntimeError(f"{name}: OpenSLR 미러 세 곳 모두에서 받지 못했습니다.")
        digest = hashlib.md5()
        with open(tar, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 22), b""):
                digest.update(block)
        if digest.hexdigest() != LIBRISPEECH_MD5[name]:
            tar.unlink()
            raise RuntimeError(f"{name}: md5가 OpenSLR 공개값과 다릅니다. 받은 파일을 지웠으니 이 셀을 다시 실행하세요.")
    # 임시 폴더에 다 푼 뒤 옮긴다: 중간에 끊긴 압축 해제가 '이미 풀려 있음'으로 보이지 않게
    staging = Path(DATA_ROOT, f".extract-{name}")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    sh(f'tar -xzf "{tar}" -C "{staging}"')
    Path(DATA_ROOT, "LibriSpeech").mkdir(exist_ok=True)
    for item in Path(staging, "LibriSpeech").iterdir():  # <name>/ 와 SPEAKERS.TXT 등
        target = Path(DATA_ROOT, "LibriSpeech", item.name)
        if not target.exists():
            item.rename(target)
    shutil.rmtree(staging, ignore_errors=True)
    if tar != cached:
        if shutil.disk_usage(DRIVE_ROOT).free > tar.stat().st_size + 2e9:  # Drive 무료 용량 15 GB를 넘기지 않게
            shutil.copy2(tar, cached.with_name(cached.name + ".part"))
            os.replace(cached.with_name(cached.name + ".part"), cached)
        else:
            print(f"Drive 여유 공간이 부족해 {name} 압축 파일을 캐시하지 않습니다 (다음 세션에 다시 받습니다).")
        tar.unlink()

SETS = [f"--set {o}" for o in OVERRIDES.split()]
DEV_CLEAN_ONLY = SPEAKERS == "dev-clean-only" and not OWN_TRAIN_ROOT
if DEV_CLEAN_ONLY:
    SETS.append("--set data.train_root=LibriSpeech/dev-clean")
RUN_NAME = CONFIG + ("-devclean" if DEV_CLEAN_ONLY else "") + (f"-{RUN_TAG}" if RUN_TAG else "")
RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}"
ARGS = (f'--config configs/{CONFIG}.json --data-root "{DATA_ROOT}" --run-dir "{RUN_DIR}" '
        f'--model-cache "{DRIVE_ROOT}/models/ecapa" --bank-mirror "{DRIVE_ROOT}/rirbanks" '
        + ("--allow-code-change " if ALLOW_CODE_CHANGE else "") + " ".join(SETS))

if sh(f"{sys.executable} train.py {ARGS} --check-banks", check=False) == 3:
    # 평가셋과 같은 pyroomacoustics 0.8.5. Linux 휠이 없어 Git 태그에서 빌드한다
    # (macOS에서는 태그 빌드가 휠과 RIR이 비트 단위로 같았다. Linux 빌드는 비교하지 않았다).
    sh('pip -q install -c /content/constraints.txt '
       '"pyroomacoustics @ git+https://github.com/LCAV/pyroomacoustics@v0.8.5"')
sh(f"{sys.executable} train.py {ARGS} --prepare-only")

## 4. 학습

학습하지 않는 시스템(`voice_only`, `space_only`, `naive_fusion`)은 바로 검증으로 넘어갑니다.
제안 모델은 `train.val_every` 걸음마다 검증 세트로 화자 정확도(Spk Acc)를 재고, 가장 좋은 가중치를
`best.pt`로 남깁니다. 끊긴 뒤 다시 실행하면 `ckpt/LATEST.json`에 적힌 가장 최근의 온전한 체크포인트에서 이어 갑니다.

로그 읽는 법:
- `loss_spk`·`acc_spk`: 화자 손실. 제안 모델에서는 게이트만 이 손실로 배웁니다.
- `loss_pos`·`acc_pos`: 위치 손실. 공간 경로는 화자가 아니라 음원 위치를 가르도록 이 손실로만 배웁니다.
- `gate_M2`~`gate_M4`: 마이크 수별로 공간 단서에 두는 비중(0~1)입니다. 마이크 1개에서는 구조상 0입니다.
- `theta_learned`: 학습된 유사도 척도가 가리키는 슬롯 임계값입니다.
- `data_sec`과 `step_sec`: `data_sec`이 `step_sec`보다 크면 GPU가 데이터를 기다리고 있다는 뜻입니다.

In [ ]:
sh(f"{sys.executable} train.py {ARGS}")

## 5. 결과 확인

검증 세트는 본 평가셋의 주 조건처럼 화자 2명(가까운 쪽 1명, 먼 쪽 1명), 턴마다 위치 사본(다시 말하는 화자는 다른 사본),
확산 잡음 SNR 5·10·20 dB로 만듭니다. 임계값(시스템별·마이크 수별)과 최고 체크포인트(마이크 2개 이상)를 이 세트로 정합니다.
정지·무잡음 조건에서는 학습 없는 단순 결합도 이미 상한 가까이 가서, 학습의 이득을 잴 여지가 없기 때문입니다.

- **Spk Acc**: 화자 슬롯과 실제 화자를 가장 잘 맞게 짝지어 잰 검증 정확도입니다. 정답은 공식 규칙(시점 이후 0.25초)을 따릅니다.
- **2인 공식**: 같은 케이스를 현재 채점 계약(세션당 2명, 슬롯 2개)대로 `evaluate.py --track 2`로 직접 채점한 값입니다.
- **causal 상한**: 들은 화자를 이어 부르는 오라클의 값입니다. 화자가 바뀌는 순간의 한 시점을 놓치는 만큼 1보다 낮습니다.
  라벨 규칙의 활성 기준으로 "들은 화자"를 정한 근사치라, 조용한 발화 시작을 듣는 시스템은 이 값을 조금 넘을 수 있습니다.
- **진단**(제안 모델, 같은 세션으로 짝지어 비교):
  - 목소리 전용 대비 이득
  - 학습된 공간 경로만 쓴 성능
  - 게이트를 마이크 수별 고정 가중치로 바꾼 성능
  - 정지·무잡음 조건 (본 평가셋의 정지 조건과 같은 꼴, 임계값은 이 조건에서 다시 정합니다)
  - 모든 화자를 한 위치에 겹친 조건: 여기서 공간 경로만으로는 우연 수준이어야 하고, 전체 모델이 목소리 전용보다 나빠지면 안 됩니다.

In [ ]:
summary = json.loads(Path(RUN_DIR, "summary.json").read_text(encoding="utf-8"))
official = (summary.get("official_n2") or {}).get("speaker_attribution_accuracy")
print(f"{summary['run']}  ({summary['system']}, {summary['trained_steps']} steps, best at {summary.get('best_step')})")
print(f"Spk Acc(cap4) {summary['spk_acc']:.4f} | 2인 공식(cap2, evaluate.py) {official} | causal 상한 {summary['causal_ceiling_by_N']}")
print()
gate = summary.get("gate_by_M", {})
cap2_by_m = summary.get("spk_acc_n2_cap2_by_M", {})
theta_off = summary.get("theta_official_by_M", summary["theta_by_M"])
print(f"{'마이크 수':>8}{'2인 공식(cap2)':>14}{'내부(cap4)':>11}{'교대 직후':>10}{'턴 안':>8}{'공식 임계값':>10}" + (f"{'gate':>8}" if gate else ""))
for M in sorted(summary["spk_acc_by_M"]):
    row = (f"{M:>8}{cap2_by_m.get(M, summary['spk_acc_by_M'][M]):>14.4f}{summary['spk_acc_by_M'][M]:>11.4f}"
           f"{summary.get('spk_acc_after_change_by_M', {}).get(M, float('nan')):>10.4f}"
           f"{summary.get('spk_acc_steady_by_M', {}).get(M, float('nan')):>8.4f}{theta_off[M]:>10.2f}")
    print(row + (f"{gate.get(M, float('nan')):>8.3f}" if gate else ""))
diag = summary.get("diagnostics")
if diag:
    print()
    print("진단 (같은 검증 세션, 마이크 수별 Spk Acc)")
    for name, key in (("목소리 전용", "voice_only_by_M"), ("목소리 전용 대비 이득", "paired_gain_over_voice_by_M"),
                      ("학습된 공간 경로만", "learned_space_only_by_M"), ("고정 게이트", "fixed_gate_by_M")):
        print(f"  {name:<14}", diag[key])
    print("  정지·무잡음 조건 이득  ", diag["static"]["paired_gain_over_voice_by_M"])
    print("  한 위치 조건 이득      ", diag["colocated"]["paired_gain_over_voice_by_M"])
    print("  한 위치 조건, 공간 경로만의 화자 분리", diag["colocated"]["spatial_part_talker_acc_by_M"],
          f"(우연 수준 약 {diag['colocated']['chance_approx']})")

metrics = [json.loads(line) for line in open(Path(RUN_DIR, "metrics.jsonl"), encoding="utf-8")]
train_rows = [m for m in metrics if m.get("kind") == "train"]
val_rows = [m for m in metrics if m.get("kind") == "val"]
if train_rows:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for key in ("loss_spk", "loss_pos"):
        ax[0].plot([m["step"] for m in train_rows if key in m], [m[key] for m in train_rows if key in m], label=key)
    ax[0].set_title("train loss")
    ax[0].set_xlabel("step")
    ax[0].legend(fontsize=8)
    for M in "1234":
        pts = [(m["step"], m[f"gate_M{M}"]) for m in train_rows if f"gate_M{M}" in m]
        if pts:
            ax[1].plot(*zip(*pts), label=f"gate M={M}")
    if val_rows:
        ax[1].plot([m["step"] for m in val_rows], [m.get("spk_acc_n2_cap2", m["spk_acc"]) for m in val_rows], "k.-", label="val Spk Acc (cap2)")
    ax[1].set_xlabel("step")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. 비교표 — 지금까지 끝난 모든 시스템

Drive의 `vcsasr/runs/` 아래 요약을 모아 마이크 수별 Spk Acc를 나란히 봅니다. 가설 H1(마이크 1개에서 2개로 갈 때
가장 크게 개선)을 보는 표입니다. `SPEAKERS`가 같은 실행끼리는 검증 세트가 같아(같은 시드) 바로 비교할 수 있습니다.
`smoke`·`smoke_real`과 `-devclean` 실행은 검증 세트가 달라 표에서 따로 표시합니다.

In [ ]:
rows = []
for path in sorted(Path(DRIVE_ROOT, "runs").glob("*/summary.json")):
    s = json.loads(path.read_text(encoding="utf-8"))
    rows.append((path.parent.name, s))
print(f"{'run':<26}" + "".join(f"{'M=' + M:>9}" for M in "1234") + f"{'전체':>9}{'2인 공식':>10}")
for name, s in rows:
    by_m = s.get("spk_acc_n2_cap2_by_M") or s.get("spk_acc_by_M", {})
    official = (s.get("official_n2") or {}).get("speaker_attribution_accuracy")
    other_set = name.startswith("smoke") or name.endswith("-devclean")
    print(f"{name + (' *' if other_set else ''):<26}" + "".join(f"{by_m.get(M, float('nan')):>9.4f}" for M in "1234")
          + f"{s.get('spk_acc_n2_cap2', s.get('spk_acc', float('nan'))):>9.4f}{(official if official is not None else float('nan')):>10.4f}")
print("* 검증 세트가 다른 실행 (smoke 계열 또는 dev-clean 전용)")

## 7. 요약 블록 — 이 출력을 그대로 붙여 넣어 주세요

In [ ]:
from train import SUMMARY_KEYS

block = {k: summary.get(k) for k in SUMMARY_KEYS if summary.get(k) is not None}
print("===== VC-SASR RUN SUMMARY (paste this back) =====")
print(json.dumps(block, ensure_ascii=False))
print("===== END =====")

## 8. 본 평가셋(`eval-t2`, `eval-t2-static`) 추론 및 채점 (선택)

현재 `CONFIG`의 검증셋 임계값(`summary.json`)과 최고 가중치(`best.pt`)를 그대로 고정해 본 평가셋 200세션(주 조건·정지 조건 각 1,200케이스)에서 `eval_t2.py`를 실행하고 공식 `Spk Acc`·`cpWER`을 계산합니다.
- 처음 실행할 때만 본 평가셋 오디오(약 1 GB)와 동결 ASR(`facebook/wav2vec2-base-960h`)을 내려받습니다.
- 전사 캐시(`vcsasr/asr_cache/`)와 결과(`vcsasr/eval/<RUN_NAME>/`)는 모두 Drive에 남으므로, 다음 시스템부터는 이미 전사된 구간을 재사용해 훨씬 빠르게 끝납니다.

In [ ]:
RUN_EVAL_T2 = False  # @param {type:"boolean"}

if RUN_EVAL_T2:
    if not Path(REPO_DIR, "data/eval-t2/audio/t2_0000.wav").is_file():
        sh(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/eval-dev/audio/'")
        sh(f'git -C "{REPO_DIR}" checkout -q HEAD')
    sh("make asr-cache")
    eval_out = f"{DRIVE_ROOT}/eval/{RUN_NAME}"
    asr_cache_dir = f"{DRIVE_ROOT}/asr_cache"
    comp_flag = ""
    if CONFIG == "naive_fusion":
        v_eval = Path(DRIVE_ROOT, "eval", "voice_only" + ("-devclean" if DEV_CLEAN_ONLY else ""))
        s_eval = Path(DRIVE_ROOT, "eval", "space_only" + ("-devclean" if DEV_CLEAN_ONLY else ""))
        if (v_eval / "eval-t2/embeddings.npz").is_file() and (s_eval / "eval-t2/embeddings.npz").is_file():
            comp_flag = f'--components-from "{v_eval}" "{s_eval}"'
    sh(f'{sys.executable} eval_t2.py --run-dir "{RUN_DIR}" --out "{eval_out}" '
       f'--device {"cuda" if GPU else "cpu"} --model-cache "{DRIVE_ROOT}/models/ecapa" '
       f'--asr-cache "{asr_cache_dir}" {comp_flag}')
    eval_summary = {}
    for set_name in ("eval-t2", "eval-t2-static"):
        for score_path in sorted(Path(eval_out, set_name).glob("*/score.json")):
            rep = json.loads(score_path.read_text(encoding="utf-8"))
            key = f"{set_name}/{score_path.parent.name}"
            eval_summary[key] = {
                "overall": {k: rep["metrics"].get(k) for k in ("speaker_attribution_accuracy", "cpwer", "wer")},
                "by_M": {m: {k: v["metrics"].get(k) for k in ("speaker_attribution_accuracy", "cpwer", "wer")}
                         for m, v in rep.get("breakdown", {}).get("by_n_channels", {}).items()},
                "frame_optimal_spk_acc_by_M": rep.get("frame_optimal_spk_acc_by_M"),
                "eval_t2_run": rep.get("eval_t2_run"),
            }
    print("===== VC-SASR EVAL-T2 SUMMARY (paste this back) =====")
    print(json.dumps({"run": RUN_NAME, "results": eval_summary}, ensure_ascii=False))
    print("===== END =====")